<a href="https://colab.research.google.com/github/eltahir64-spec/Agentic-AI-memory-Implementation/blob/main/NM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Project Setup: Environment and Database Initialization

First, let's install all the necessary Python libraries for our project. This includes `crewai` for agent orchestration, `fastapi` and `uvicorn` for the REST API layer, `pyngrok` to expose our Colab server, `pandas` for log processing, and `openai` as our LLM backend.

In [9]:
# Install required libraries
!pip install crewai fastapi uvicorn[standard] pyngrok pandas openai scikit-learn

print("All required libraries installed.")

<frozen posixpath>:82: RuntimeWarning: coroutine 'Server.serve' was never awaited


All required libraries installed.


Next, we'll set up our SQLite database. SQLite is an excellent choice for an embedded, serverless database in Colab. We'll define the schema for the six tables as specified in Layer 4 of your requirements: `users`, `auth_logs`, `network_events`, `email_events`, `detected_incidents`, and `response_actions`.

In [10]:
import sqlite3

DATABASE_NAME = 'security_log_db.sqlite'

def create_database_schema():
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()

    # Table: users (Example structure for user management within the system)
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT NOT NULL UNIQUE,
            email TEXT NOT NULL UNIQUE,
            role TEXT DEFAULT 'analyst'
        )
    ''')

    # Table: auth_logs (For Active Directory and Microsoft Entra ID logs)
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS auth_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
            source TEXT NOT NULL,
            event_id TEXT,
            username TEXT,
            action TEXT,
            status TEXT,
            ip_address TEXT,
            details TEXT
        )
    ''')

    # Table: network_events (For AWS CloudTrail/GuardDuty, Check Point Firewall, Internet filtering)
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS network_events (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
            source TEXT NOT NULL,
            event_type TEXT,
            protocol TEXT,
            src_ip TEXT,
            dst_ip TEXT,
            src_port INTEGER,
            dst_port INTEGER,
            action TEXT,
            details TEXT
        )
    ''')

    # Table: email_events (For Proofpoint email security)
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS email_events (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
            source TEXT NOT NULL,
            sender TEXT,
            recipient TEXT,
            subject TEXT,
            action TEXT,
            threat_level TEXT,
            details TEXT
        )
    ''')

    # Table: detected_incidents (To store incidents identified by AI agents)
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS detected_incidents (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
            incident_id TEXT NOT NULL UNIQUE,
            severity TEXT,
            category TEXT,
            description TEXT,
            status TEXT DEFAULT 'open',
            assigned_to INTEGER,
            FOREIGN KEY (assigned_to) REFERENCES users(id)
        )
    ''')

    # Table: response_actions (To store recommended and executed response actions)
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS response_actions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
            incident_id TEXT NOT NULL,
            recommended_action TEXT,
            executed_action TEXT,
            status TEXT DEFAULT 'pending',
            FOREIGN KEY (incident_id) REFERENCES detected_incidents(incident_id)
        )
    ''')

    conn.commit()
    conn.close()
    print(f"Database '{DATABASE_NAME}' and tables created/verified successfully.")

# Execute the function to create the database and schema
create_database_schema()

Database 'security_log_db.sqlite' and tables created/verified successfully.


Now that our environment is set up and the database schema is in place, our next step will be to implement **Layer 2: Ingestion Pipeline**. This will involve setting up a FastAPI application to act as a REST API collector and defining endpoints for each log source to push data into our raw log queue, which we will simulate first and then persist to the database.

In [11]:
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import sqlite3
import json
import threading
import time
import queue
from datetime import datetime
import nest_asyncio # Import nest_asyncio
from pyngrok import ngrok # Ensure ngrok is imported once

# IMPORTANT: If you encounter 'RuntimeError: asyncio.run() cannot be called from a running event loop' again,
# please restart your Colab runtime (Runtime -> Restart runtime...) and then run all cells from the beginning.
nest_asyncio.apply() # Apply the patch to allow nested asyncio event loops

# Initialize FastAPI app
app = FastAPI()

# Global queue for raw logs
raw_log_queue = queue.Queue()

# Database name from earlier setup
DATABASE_NAME = 'security_log_db.sqlite'

# Pydantic models for incoming log data
class AuthLog(BaseModel):
    source: str
    event_id: str | None = None
    username: str | None = None
    action: str | None = None
    status: str | None = None
    ip_address: str | None = None
    details: str | None = None

class NetworkEvent(BaseModel):
    source: str
    event_type: str | None = None
    protocol: str | None = None
    src_ip: str | None = None
    dst_ip: str | None = None
    src_port: int | None = None
    dst_port: int | None = None
    action: str | None = None
    details: str | None = None

class EmailEvent(BaseModel):
    source: str
    sender: str | None = None
    recipient: str | None = None
    subject: str | None = None
    action: str | None = None
    threat_level: str | None = None
    details: str | None = None

# --- API Endpoints for Log Collection ---

@app.post("/ingest/auth")
async def ingest_auth_log(log: AuthLog):
    raw_log_queue.put({"type": "auth", "data": log.dict()})
    print(f"Received Auth Log: {log.dict()}") # For demonstration
    return {"message": "Auth log received", "status": "pending_processing"}

@app.post("/ingest/network")
async def ingest_network_event(event: NetworkEvent):
    raw_log_queue.put({"type": "network", "data": event.dict()})
    print(f"Received Network Event: {event.dict()}") # For demonstration
    return {"message": "Network event received", "status": "pending_processing"}

@app.post("/ingest/email")
async def ingest_email_event(event: EmailEvent):
    raw_log_queue.put({"type": "email", "data": event.dict()})
    print(f"Received Email Event: {event.dict()}") # For demonstration
    return {"message": "Email event received", "status": "pending_processing"}


# --- Ngrok Setup to expose FastAPI locally ---

# You need to set your ngrok auth token to avoid usage limits and authentication errors.
# Get your token from https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token("3BsMVVRTkiTRyuMBd6p6NATiGkQ_6VrCKwd2hLmG7QwpFxygz") # <<< REPLACE WITH YOUR NGROK AUTH TOKEN

def run_fastapi_server(): # Renamed to avoid confusion with the main thread
    # Use a specific port for uvicorn
    # uvicorn.run will start its own event loop within this thread
    uvicorn.run(app, host="0.0.0.0", port=8000)

def start_ngrok_tunnel(): # Renamed to differentiate from original start_ngrok
    try:
        # Ensure any existing ngrok tunnels are closed
        ngrok.kill()
        # Open a tunnel to the FastAPI app running on port 8000
        public_url = ngrok.connect(8000).public_url
        print(f"FastAPI running on public URL: {public_url}")
        # Store the public URL for later use or display
        global NGROK_PUBLIC_URL
        NGROK_PUBLIC_URL = public_url
    except Exception as e:
        print(f"Error starting ngrok: {e}")

# --- Raw Log Processor: Parses, Normalizes, and Persists to DB ---
def process_raw_logs():
    print("Starting raw log processor...")
    while True:
        if not raw_log_queue.empty():
            log_entry = raw_log_queue.get()
            log_type = log_entry["type"]
            log_data = log_entry["data"]
            print(f"Processing raw log (type: {log_type}): {log_data}")

            try:
                conn = sqlite3.connect(DATABASE_NAME)
                cursor = conn.cursor()
                current_timestamp = datetime.now().isoformat()

                if log_type == "auth":
                    cursor.execute('''
                        INSERT INTO auth_logs (timestamp, source, event_id, username, action, status, ip_address, details)
                        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
                    ''', (
                        current_timestamp,
                        log_data.get('source'),
                        log_data.get('event_id'),
                        log_data.get('username'),
                        log_data.get('action'),
                        log_data.get('status'),
                        log_data.get('ip_address'),
                        log_data.get('details')
                    ))
                    print(f"  -> Auth log inserted into DB: {log_data.get('source')} - {log_data.get('username')}")

                elif log_type == "network":
                    cursor.execute('''
                        INSERT INTO network_events (timestamp, source, event_type, protocol, src_ip, dst_ip, src_port, dst_port, action, details)
                        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                    ''', (
                        current_timestamp,
                        log_data.get('source'),
                        log_data.get('event_type'),
                        log_data.get('protocol'),
                        log_data.get('src_ip'),
                        log_data.get('dst_ip'),
                        log_data.get('src_port'),
                        log_data.get('dst_port'),
                        log_data.get('action'),
                        log_data.get('details')
                    ))
                    print(f"  -> Network event inserted into DB: {log_data.get('source')} - {log_data.get('src_ip')}")

                elif log_type == "email":
                    cursor.execute('''
                        INSERT INTO email_events (timestamp, source, sender, recipient, subject, action, threat_level, details)
                        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
                    ''', (
                        current_timestamp,
                        log_data.get('source'),
                        log_data.get('sender'),
                        log_data.get('recipient'),
                        log_data.get('subject'),
                        log_data.get('action'),
                        log_data.get('threat_level'),
                        log_data.get('details')
                    ))
                    print(f"  -> Email event inserted into DB: {log_data.get('source')} - {log_data.get('sender')}")

                conn.commit()
                conn.close()

            except sqlite3.Error as e:
                print(f"Database error during log processing: {e}")
                if conn:
                    conn.rollback()
                    conn.close()
            except Exception as e:
                print(f"An unexpected error occurred during log processing: {e}")

            # Simulate some work
            time.sleep(0.1)
        else:
            time.sleep(1) # Wait if queue is empty


# Run FastAPI and Ngrok in separate threads
if __name__ == '__main__':
    # Start the raw log processor in a separate thread
    log_processor_thread = threading.Thread(target=process_raw_logs)
    log_processor_thread.daemon = True # Allow the main program to exit even if this thread is running
    log_processor_thread.start()

    # Start Ngrok in a separate thread
    ngrok_thread = threading.Thread(target=start_ngrok_tunnel) # Use new function name
    ngrok_thread.daemon = True
    ngrok_thread.start()

    # Start FastAPI in its own separate thread
    # This is the crucial change to avoid the asyncio.run() conflict in the main thread
    fastapi_thread = threading.Thread(target=run_fastapi_server) # Use new function name
    fastapi_thread.daemon = True
    fastapi_thread.start()

    # Keep the main thread alive for a short period or indefinitely if needed for interaction
    # In a Colab environment, the cell usually finishes execution, but the daemon threads continue.
    # To see the output of ngrok and ensure the server has time to start, a small sleep can be added.
    time.sleep(5) # Give some time for ngrok and FastAPI to initialize and print their URLs
    print("FastAPI and Ngrok threads started. Main cell execution complete.")
    # The cell will complete, but the daemon threads for FastAPI and Ngrok will continue running.

Starting raw log processor...


INFO:     Started server process [6805]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


FastAPI running on public URL: https://nonferocious-unpunctate-georgianna.ngrok-free.dev
FastAPI and Ngrok threads started. Main cell execution complete.


## Layer 2: Ingestion Pipeline - FastAPI Setup and Log Collectors

We will now set up the FastAPI application to act as the REST API collector for our log sources. We'll define a basic FastAPI app and endpoints for each log type. For now, these endpoints will simply receive the data and confirm receipt. Later, we'll integrate the raw log queue and database persistence.